# Deep Ensembles on GPU, through main's shared pipeline (CISC 473 P23)
For each ensemble seed `s` this runs `pipeline.train_with_noise(DeepEnsemble(n=5), noise_ratio=0, seed=s)` (members have seeds `s*100 + i`:
seed 0 -> 0-4, seed 1 -> 100-104, seed 2 -> 200-204), then main's `evaluate_conformal`, then the add-on dumper (per-member + SVHN-test probabilities).
It zips checkpoints, main-format results and add-on probabilities into `de_gpu_ensemble_seeds_<seeds>.zip` (refreshed every ~5 minutes while training).

Split across notebooks by editing `SEEDS` in the second cell, e.g. `[0, 1]` in one notebook and `[2]` in another.
Works on Python 3.10-3.12 with the preinstalled torch / numpy / scipy; nothing is pinned or reinstalled.

In [ ]:
# Cell 1: fail fast
import sys, torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda build", torch.version.cuda, "| cuda available", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Kaggle Settings > Accelerator > GPU (T4/P100); Colab Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 2: the only thing you edit
SEEDS = [0, 1, 2]           # ensemble seeds. Notebook A: [0, 1]   Notebook B: [2]
REPO_URL = "https://github.com/spy772/CISC-473-P23.git"
BRANCH = "de-final"
assert all(s in (0, 1, 2) for s in SEEDS)

In [ ]:
# Cell 3: clone, install only what is missing (unpinned), check data + internet
import os, subprocess, importlib.util
if not os.path.exists("CISC-473-P23"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "CISC-473-P23"], check=True)
WORK = os.getcwd()
os.chdir("CISC-473-P23")
missing = [m for m in ("numpy", "scipy", "torchvision") if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)
print("missing packages installed:", missing or "none", "| cwd:", os.getcwd())
assert os.path.exists("data/cifar-10-batches-py/data_batch_1"), "CIFAR-10 batches should be in the repo"
# imports the shared pipeline, downloads the SVHN test split, and loads the torch.hub ResNet-20 definition (needs internet)
subprocess.run([sys.executable, "-c", "import sys; sys.path.insert(0,'.'); sys.path.insert(0,'src'); import pipeline, dump_member_probs as d; "
                "print('SVHN test', d.load_svhn_test().shape); pipeline.models._hub_resnet20(); print('pipeline + hub model OK')"], check=True)

In [ ]:
# Cell 4: train -> evaluate -> dump, one ensemble seed at a time; zip refreshed every 5 min so a disconnect keeps finished members.
# Member checkpoints are saved as soon as each member finishes; re-running this cell skips finished members (main's resume).
import glob, zipfile, time
ZIP = os.path.join(WORK, "de_gpu_ensemble_seeds_%s.zip" % "_".join(map(str, SEEDS)))   # outside the repo -> shows up in Kaggle Output

def make_zip():
    files = []
    for s in SEEDS:
        tag = "deep_ensemble_noise0_seed%d" % s
        files += glob.glob("models/noise_testing/%s_m*.pt" % tag)
        files += glob.glob("results/noise_testing/%s.json" % tag) + glob.glob("results/noise_testing/%s.npz" % tag)
        files += glob.glob("results/noise_testing/de_addon/%s/*" % tag)
    with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for f in files:
            z.write(f)
    print("  zip updated:", ZIP, len(files), "files,", round(os.path.getsize(ZIP) / 1e6, 1), "MB", flush=True)

for s in SEEDS:
    t0 = time.time()
    p = subprocess.Popen([sys.executable, "src/de_train.py", "--seeds", str(s)])
    while p.poll() is None:
        time.sleep(300)
        make_zip()
    assert p.returncode == 0, "training failed for ensemble seed %d" % s
    make_zip()
    print("ensemble seed", s, "finished in %.1f min" % ((time.time() - t0) / 60))

In [ ]:
# Cell 5: summary
import json
print("ens_seed member  member_seed  train_min  test_acc")
for s in SEEDS:
    m = json.load(open("results/noise_testing/de_addon/deep_ensemble_noise0_seed%d/meta.json" % s))
    for mm in m["members"]:
        print("%8d %6d %12d %10.1f %9.4f" % (s, mm["member"], mm["member_seed"], mm["train_minutes"], mm["test_acc"]))
print("Download:", ZIP)
# Colab only: from google.colab import files; files.download(ZIP)